# Library 00 – Project settings, logging and table helpers

This notebook holds **every setting of the study in one place**: folder paths, the study window, the
format names and the random seed. It also has two small helpers that every stage needs: a logger that
writes to `outputs/logs/`, and functions that read and write CSV tables with one date format.

It only *defines* things. Stage notebooks load it with `%run ./library/lib_00_settings.ipynb`.

In [ ]:
# Standard-library and third-party imports shared by all notebooks.
import hashlib
import json
import logging
import statistics
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Iterator

import networkx as nx
import numpy as np
import pandas as pd

## 1. Folder paths

The notebooks may be opened from the `notebooks/` folder or from the project root. We look upwards from the
current folder until we find the instructions file, and treat that folder as the project root.

In [ ]:
def find_project_root(start_folder: Path) -> Path:
    """Walk up from ``start_folder`` until the folder holding the project instructions is found."""
    for candidate_folder in [start_folder, *start_folder.parents]:
        if (candidate_folder / "INSTRUCTIONS_England_Batting_Network.md").exists():
            return candidate_folder
    raise FileNotFoundError("Could not find the project root (INSTRUCTIONS_England_Batting_Network.md).")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Raw data: the Cricsheet zip archive is never changed. The extracted folder is a cache rebuilt from it.
RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
RAW_ARCHIVE_PATH = RAW_DATA_FOLDER / "england_male_json.zip"
RAW_MATCH_FOLDER = RAW_DATA_FOLDER / "england_male_json"
ARCHIVE_SOURCE_URL = "https://cricsheet.org/downloads/england_male_json.zip"

# Derived data and outputs.
INTERIM_FOLDER = PROJECT_ROOT / "data" / "interim"        # manifest and file audit
PROCESSED_FOLDER = PROJECT_ROOT / "data" / "processed"    # clean tables used by later stages
TABLES_FOLDER = PROJECT_ROOT / "outputs" / "tables"       # measure and validation tables
GRAPHML_FOLDER = PROJECT_ROOT / "outputs" / "graphml"     # one GraphML file per format (for Gephi)
FIGURES_FOLDER = PROJECT_ROOT / "outputs" / "figures"     # PNG figures used in the report
LOGS_FOLDER = PROJECT_ROOT / "outputs" / "logs"           # one log file per stage
REPORT_FOLDER = PROJECT_ROOT / "report"
TEMPLATE_PATH = PROJECT_ROOT / "docs" / "reference" / "Network_Analysis_Project_Template.docx"

## 2. Study definition

These values decide which matches belong to the study. Changing them here changes every later stage.

In [ ]:
TEAM_NAME = "England"
TEAM_GENDER = "male"
TEAM_TYPE = "international"

# A match is included when its FIRST day lies inside this closed interval.
STUDY_START_DATE = date(2020, 1, 1)
STUDY_END_DATE = date(2026, 10, 29)

# Cricsheet calls the match type "Test", "ODI", "T20" (or "IT20"). We use these labels everywhere.
CRICSHEET_TO_FORMAT = {"Test": "Test", "ODI": "ODI", "T20": "T20I", "IT20": "T20I"}
FORMATS = ("Test", "ODI", "T20I")
FORMAT_FILE_TAG = {"Test": "test", "ODI": "odi", "T20I": "t20i"}   # used in file names

## 3. Reproducibility settings

In [ ]:
RANDOM_SEED = 42                 # used by Louvain, the random reference graphs and the layout
LOUVAIN_RESOLUTION = 1.0         # standard modularity (no resolution tuning)
NUMBER_OF_RANDOM_GRAPHS = 200    # random graphs used for the small-world comparison
LOUVAIN_STABILITY_RUNS = 100     # seeds 0..99 used to check how stable the communities are

DATE_TEXT_FORMAT = "%d-%m-%Y"    # every date in an exported table is written as DD-MM-YYYY


def create_output_folders() -> None:
    """Create every folder the pipeline writes to (no error if they already exist)."""
    for output_folder in (INTERIM_FOLDER, PROCESSED_FOLDER, TABLES_FOLDER, GRAPHML_FOLDER, FIGURES_FOLDER, LOGS_FOLDER):
        output_folder.mkdir(parents=True, exist_ok=True)


create_output_folders()

## 4. Logging

Each stage writes what it did to the screen and to `outputs/logs/<stage>.log`, so a run leaves a written record.

In [ ]:
def get_stage_logger(stage_name: str) -> logging.Logger:
    """Return a logger that prints to the notebook and writes to ``outputs/logs/<stage_name>.log``."""
    stage_logger = logging.getLogger(f"batnet.{stage_name}")
    stage_logger.setLevel(logging.INFO)
    stage_logger.propagate = False
    # Remove old handlers so that re-running a cell does not print every line twice.
    for old_handler in list(stage_logger.handlers):
        stage_logger.removeHandler(old_handler)
        old_handler.close()
    line_format = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    file_handler = logging.FileHandler(LOGS_FOLDER / f"{stage_name}.log", mode="w", encoding="utf-8")
    screen_handler = logging.StreamHandler()
    for handler in (file_handler, screen_handler):
        handler.setFormatter(line_format)
        stage_logger.addHandler(handler)
    return stage_logger

## 5. Reading and writing tables

All CSV files are UTF-8. Date columns are written as `DD-MM-YYYY` and read back as Python dates. ID columns
are always read as text, so a Cricsheet ID such as `1022347` is never turned into a number.

In [ ]:
DATE_COLUMNS = ("match_date", "start_date", "end_date", "first_date", "last_date")
ID_COLUMNS = ("match_id", "player_id", "player_1_id", "player_2_id",
              "start_delivery_id", "end_delivery_id", "delivery_id")


def write_table(table: pd.DataFrame, output_path: Path) -> None:
    """Save a table as CSV with DD-MM-YYYY dates."""
    table_to_save = table.copy()
    for column_name in DATE_COLUMNS:
        if column_name in table_to_save.columns:
            table_to_save[column_name] = pd.to_datetime(table_to_save[column_name]).dt.strftime(DATE_TEXT_FORMAT)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    table_to_save.to_csv(output_path, index=False, encoding="utf-8")


def read_table(input_path: Path) -> pd.DataFrame:
    """Read a CSV written by :func:`write_table`; dates come back as ``datetime.date`` values."""
    header_columns = pd.read_csv(input_path, nrows=0).columns
    text_columns = {column_name: str for column_name in ID_COLUMNS if column_name in header_columns}
    table = pd.read_csv(input_path, dtype=text_columns)
    for column_name in DATE_COLUMNS:
        if column_name in table.columns:
            table[column_name] = pd.to_datetime(table[column_name], format=DATE_TEXT_FORMAT).dt.date
    return table


def markdown_table(table: pd.DataFrame) -> str:
    """Turn a small table into Markdown text (used in the data-quality report)."""
    header_line = "| " + " | ".join(map(str, table.columns)) + " |"
    divider_line = "|" + "---|" * len(table.columns)
    body_lines = ["| " + " | ".join(map(str, row)) + " |" for row in table.itertuples(index=False)]
    return "\n".join([header_line, divider_line, *body_lines])